# 08 — Kaggle publication

Build the provenance manifest, Kaggle model metadata, and a release record, then publish the adapter as a new Kaggle model variation — only when the release gates pass. Nothing is published from this notebook unless `verify_release.py` exits 0.

In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())

In [ ]:
from src.publishing.provenance import build_provenance, validate_provenance
provenance = build_provenance(
    model="Qwen3.8-27B-akarui",
    base_model="Qwen/Qwen3.8-27B",
    base_revision="37561816e66b5b9153b1b64c10968c18c1e6f4a0",
    training_method="QLoRA",
    training_date="2026-11-18",
    training_dataset_version="v0",
    training_dataset_hash="FILL_FROM_DATASET_MANIFEST",
    training_config_hash="FILL_FROM_CONFIG_HASH",
    source_repository="https://github.com/Monibee-Fudgekins/Qwen3.8-27B-akarui",
    source_commit="FILL_FROM_GIT_SHA",
    training_environment="Kaggle",
    gpu_configuration=[{"model": "L4", "count": 1, "vram_gb": 24}],
    evaluation_results={},
    artifact_hashes={},
    release_version="1",
)
print("validation errors (expected until hashes are filled):", validate_provenance(provenance))

In [ ]:
from src.publishing.kaggle_publisher import build_model_metadata
metadata = build_model_metadata(
    "Qwen3.8-27B-akarui", "qlora-sft",
    "Competition-solving fine-tune of Qwen3.8-27B.",
    ["qwen", "qwen3", "pytorch", "qlora", "data-science", "kaggle-competition"],
)
import json
print(json.dumps(metadata, indent=2))

In [ ]:
from src.publishing.provenance import provenance_to_json
path = provenance_to_json(provenance, "releases/1-provenance.json")
print("provenance draft written to", path)
print("Fill training_dataset_hash, training_config_hash, and source_commit")
print("from the dataset manifest, config hash, and git SHA before publishing.")

In [ ]:
import subprocess, sys
# Requires real artifacts: the provenance file above (with real hashes)
# and evaluation/results.json from notebook 07. Until then this exits 1.
result = subprocess.run([sys.executable, "scripts/verify_release.py",
                         "--provenance", "releases/1-provenance.json",
                         "--evaluation", "evaluation/results.json",
                         "--config", "configs/evaluation.yaml"],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)
print("verify exit code:", result.returncode, "(0 = gates pass, publish allowed)")

In [ ]:
# Dry run first: stages metadata without uploading.
# Real publish (requires KAGGLE_TOKEN in the Kaggle notebook secrets):
#   !python scripts/publish_kaggle.py --provenance releases/1-provenance.json \
#       --evaluation evaluation/results.json --adapter output/akarui-sft-v0 \
#       --release-version 1 --model-version qlora-sft-v1 --dry-run
# Remove --dry-run only after the staged files look right.
print("Dry-run the publish script with --dry-run before any real upload.")